# 영상 → Deux BGM 제거 → 일본어 대사·타임스탬프·음성 ZIP

**코랩 GPU에서 영상을 올리고 아래 셀을 순서대로 실행하면 로컬 검수용 1차 데이터셋을 다운로드합니다.**

Mel-Band RoFormer Deux로 영상 전체의 BGM을 먼저 분리한 뒤, Vocals를 Qwen3-ASR 1.7B → Qwen3-ForcedAligner 0.6B로 인식·시간 정렬합니다. Silero VAD는 CPU에서 음성 구간을 찾습니다. 공개 모델을 익명으로 내려받으며 HF 토큰·Drive 연결·Gradio 로그인 입력이 없습니다.

1. `런타임 → 런타임 유형 변경 → T4 GPU` 이상을 선택합니다.
2. 셀 1에서 전용 Python 환경을 설치합니다. 세션 재시작은 필요 없습니다.
3. 셀 2에서 영상 하나를 업로드합니다. 기본값은 첫 오디오 트랙의 일본어입니다.
4. 셀 3에서 처리한 뒤 셀 4에서 ZIP을 다운로드합니다.
5. 로컬에서 ZIP을 풀어 `review.csv`의 시간·대사와 `wav/`를 비교합니다. 채택한 WAV와 교정한 `captions.csv`로 최종 데이터셋을 만듭니다.

**모든 화자의 인식 결과를 후보로 보존합니다.** 에키드나 식별·다른 화자 제거는 수행하지 않습니다. WAV는 Deux의 Vocals 출력에서 추출합니다. BGM 잔여와 분리 과정의 음성 손상은 검수해야 합니다. 인식되지 않은 음성은 누락될 수 있으며 정렬 실패·빈 전사는 검수 표시를 남깁니다.

ZIP 구성은 `echidna/captions.csv`, `echidna/review.csv`, `echidna/wav/*.wav`, `echidna/metadata/`, `echidna/검수안내.txt`입니다. `captions.csv`는 현재 폴더의 Irodori 학습 노트북이 읽는 `audio_file,text` 형식입니다. WAV는 Deux Vocals를 변환한 48kHz 모노 PCM16이며, 인식용 16kHz 음성을 확대해서 만들지 않습니다. Deux 원본 모델의 처리 주파수는 44.1kHz입니다. **각 STT 음성 구간 전체와 그 전체 전사를 한 쌍으로 보존하며, 마지막 단어의 정렬 시각에서 WAV를 자르지 않습니다.** 원본 영상 파일은 ZIP에 포함하지 않습니다.

코랩 런타임이 초기화되면 업로드 영상과 처리 결과가 사라집니다. 같은 런타임에서는 셀 3 재실행 시 완료한 전사·정렬을 재사용합니다.

Deux는 공식 체크포인트와 YAML을 익명으로 다운로드합니다. 배치·겹침·입력 길이는 YAML을 사용하며, STT·VAD의 기존 수치는 유지합니다. 실제 적용 설정은 `metadata/separation_info.json`에 기록됩니다.

참고: [Deux](https://huggingface.co/becruily/mel-band-roformer-deux), [audio-separator](https://github.com/nomadkaraoke/python-audio-separator), [Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR), [Silero VAD](https://github.com/snakers4/silero-vad), [Irodori-TTS](https://github.com/Aratako/Irodori-TTS).

In [ ]:
#@title 1. GPU 확인 · 전용 실행 환경 설치
import sys, subprocess, shutil, urllib.request, os, signal
from pathlib import Path

WORK_ROOT = Path('/content/echidna_dataset_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def run_process(args):
    process = subprocess.Popen([str(x) for x in args], stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'실행 실패 (exit={process.returncode}). 위 로그를 확인하세요.')
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    finally:
        process.stdout.close()

if not shutil.which('nvidia-smi'):
    raise RuntimeError('런타임 유형을 GPU로 바꾸세요.')
run_process(['nvidia-smi'])
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_process(['apt-get', '-qq', 'update'])
    run_process(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
uv = shutil.which('uv')
if not uv:
    installer = WORK_ROOT / 'uv-install.sh'
    urllib.request.urlretrieve('https://astral.sh/uv/install.sh', installer)
    run_process(['sh', installer])
    uv = str(Path.home() / '.local/bin/uv')
VENV = WORK_ROOT / '.venv'
PY = VENV / 'bin/python'
if not PY.is_file():
    run_process([uv, 'python', 'install', '3.12'])
    run_process([uv, 'venv', '--python', '3.12', VENV])
constraints = WORK_ROOT / 'constraints.txt'
constraints.write_text('torch==2.8.0\ntorchaudio==2.8.0\ntorchvision==0.23.0\ntransformers==4.57.6\n', encoding='utf-8')
base = [uv, 'pip', 'install', '--python', PY]
run_process(base + ['--index-url', 'https://download.pytorch.org/whl/cu126',
    'torch==2.8.0', 'torchaudio==2.8.0', 'torchvision==0.23.0'])
run_process(base + ['-c', constraints, 'qwen-asr==0.0.6', 'silero-vad==6.2.3',
    'soundfile', 'pandas', 'tqdm', 'audio-separator[gpu]==0.47.0',
    'audioread==3.1.0'])
print('설치 완료. 다음 셀에서 영상을 업로드하세요. 세션 재시작은 필요 없습니다.')

In [ ]:
#@title 2. 영상 업로드 · 처리 설정
audio_track = 0 #@param {type:"integer"}
dataset_name = 'echidna' #@param {type:"string"}
vad_threshold = 0.4 #@param {type:"number"}
# 같은 런타임에서 업로드한 영상을 다시 쓸 때 전체 경로를 입력하면 업로드창을 생략합니다.
existing_video_path = '' #@param {type:"string"}

import json, hashlib, re
from google.colab import files

if not re.fullmatch(r'[A-Za-z0-9_-]+', dataset_name):
    raise ValueError('dataset_name은 영문/숫자/_/-만 사용하세요.')
if not isinstance(audio_track, int) or audio_track < 0:
    raise ValueError('audio_track은 0 이상의 정수입니다.')
if not 0 < vad_threshold < 1:
    raise ValueError('VAD 임계값은 0과 1 사이여야 합니다.')
if existing_video_path.strip():
    VIDEO = Path(existing_video_path).expanduser().resolve()
    if not VIDEO.is_file():
        raise FileNotFoundError(VIDEO)
else:
    incoming = WORK_ROOT / 'uploads'
    incoming.mkdir(exist_ok=True)
    previous_cwd = Path.cwd()
    try:
        os.chdir(incoming)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError('영상 하나만 선택하세요. 셀 2를 다시 실행하세요.')
    VIDEO = (incoming / next(iter(uploaded))).resolve()
    del uploaded

probe = subprocess.run(['ffprobe', '-v', 'error', '-show_streams', '-show_format',
    '-of', 'json', str(VIDEO)], check=True, capture_output=True, text=True)
media = json.loads(probe.stdout)
tracks = [s for s in media.get('streams', []) if s['codec_type'] == 'audio']
if not tracks or audio_track >= len(tracks):
    raise ValueError(f'선택 가능한 오디오 트랙: 0~{len(tracks)-1}')
for i, track in enumerate(tracks):
    print(f"트랙 {i}: {track.get('codec_name')} / {track.get('sample_rate')}Hz / "
          f"{track.get('channels')}채널 / {track.get('tags', {}).get('language', '언어 미표기')}")
print('선택 트랙:', audio_track)
print('재사용 영상 경로:', VIDEO)
config = dict(video=str(VIDEO), audio_track=audio_track, dataset_name=dataset_name,
    vad_threshold=vad_threshold, clip_boundary='stt_speech_segment',
    separation_model='becruily/mel-band-roformer-deux',
    separation_revision='2da74427d682a3df47a774378fc24d7a1a0cdaad',
    separation_checkpoint='becruily_deux.ckpt',
    separation_config='config_deux_becruily.yaml',
    separation_package='audio-separator==0.47.0')
digest = hashlib.sha256()
with VIDEO.open('rb') as stream:
    for block in iter(lambda: stream.read(8*1024*1024), b''):
        digest.update(block)
config['video_sha256'] = digest.hexdigest()
processing = {key: value for key, value in config.items() if key != 'video'}
run_key = hashlib.sha256(json.dumps(processing, sort_keys=True).encode()).hexdigest()[:16]
RUN = WORK_ROOT / 'runs' / run_key
RUN.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN / 'config.json'
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print('결과 폴더:', RUN)

## 처리 · ZIP 생성

Deux로 영상 전체를 먼저 분리하고, Vocals에서 VAD → 전사 → 단어 시간 정렬을 수행합니다. Deux는 별도 프로세스에서 실행하여 종료 후 GPU 메모리를 반환합니다. 전사와 정렬 모델도 순서대로 올렸다가 해제합니다.

**WAV와 자막은 각 STT 음성 구간 전체를 기준으로 생성합니다.** 마지막 단어의 정렬 시각을 WAV 종료 지점으로 사용하지 않습니다. 단어별 시각은 `metadata/aligned_words.json`에 보존합니다. 기존 VAD 설정의 150ms 음성 패딩은 유지하며, 별도의 추출 패딩이나 대사 병합·분할은 추가하지 않습니다. VAD 자체가 발화를 놓치거나 Deux가 음성을 손상시키는 경우까지 자동 해결되는 것은 아닙니다.

20초가 넘는 후보와 빈 전사도 WAV와 검수 목록에 남습니다. 긴 후보는 로컬 검수 때 음성과 대사를 함께 나눠야 합니다. 화자 변경은 자동 판별하지 않습니다.

오류가 나면 셀 3을 다시 실행하세요. 같은 설정에서 완료된 Deux·전사·정렬 결과를 재사용합니다. 기존 원본 음성 처리 결과와는 별도 작업 폴더를 사용합니다. 설정을 바꾸려면 셀 2의 `existing_video_path`에 출력된 영상 경로를 넣어 실행한 뒤 셀 3을 실행합니다. 모델 다운로드는 첫 실행에 시간이 걸립니다.


In [ ]:
#@title 3. Deux BGM 제거 → 대사 인식 → 시간 정렬 → 음성 구간 전체 추출 → ZIP
DEUX_SOURCE = r'''
import os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
import sys, json, shutil
from pathlib import Path
import yaml, torch, soundfile as sf
from huggingface_hub import hf_hub_download
from audio_separator.separator import Separator

config_path, input_path, output_path = map(Path, sys.argv[1:4])
cfg = json.loads(config_path.read_text(encoding='utf-8'))
if not torch.cuda.is_available():
    raise RuntimeError('Deux 실행에 CUDA GPU가 필요합니다.')
model_dir = config_path.parent.parent.parent / 'deux_models'
model_dir.mkdir(exist_ok=True)
def download_model_file(filename):
    return hf_hub_download(repo_id=cfg['separation_model'],
        revision=cfg['separation_revision'], filename=filename,
        cache_dir=str(model_dir), token=False)
yaml_path = download_model_file(cfg['separation_config'])
with open(yaml_path, encoding='utf-8') as stream:
    model_cfg = yaml.load(stream, Loader=yaml.FullLoader)

class DeuxSeparator(Separator):
    # The stock catalogue does not contain Deux. Resolve its official pair directly.
    def download_model_files(self, model_filename):
        if model_filename != cfg['separation_checkpoint']:
            raise ValueError('선택된 Deux 체크포인트와 다릅니다.')
        checkpoint_path = download_model_file(model_filename)
        self.model_friendly_name = 'Mel-Band RoFormer Deux by becruily'
        self.model_is_uvr_vip = False
        return model_filename, 'MDXC', self.model_friendly_name, checkpoint_path, yaml_path

stem_dir = output_path.parent / 'deux_stems'
separator = DeuxSeparator(model_file_dir=str(model_dir), output_dir=str(stem_dir),
    output_format='WAV', output_single_stem='Vocals', use_soundfile=True,
    sample_rate=model_cfg['audio']['sample_rate'],
    use_autocast=bool(model_cfg['training']['use_amp']))
# No segment, overlap or batch override: audio-separator 0.47.0 reads the model YAML.
separator.load_model(model_filename=cfg['separation_checkpoint'])
if separator.torch_device.type != 'cuda':
    raise RuntimeError('Deux가 CUDA를 사용하지 않습니다.')
print('Deux 원본 YAML:', json.dumps(model_cfg['inference'], ensure_ascii=False), flush=True)
instance = separator.model_instance
settings = dict(model=cfg['separation_model'], revision=cfg['separation_revision'],
    checkpoint=cfg['separation_checkpoint'], config_file=cfg['separation_config'],
    audio_config=model_cfg['audio'], inference_config=model_cfg['inference'],
    effective_overlap=instance.overlap, effective_batch_size=instance.batch_size,
    override_model_segment_size=instance.override_model_segment_size,
    effective_precision=separator.effective_precision,
    normalization_threshold=separator.normalization_threshold,
    amplification_threshold=separator.amplification_threshold)
print('Deux 실행 설정:', json.dumps(settings, ensure_ascii=False), flush=True)
outputs = separator.separate(str(input_path), custom_output_names={'Vocals': 'deux_vocals'})
if len(outputs) != 1:
    raise RuntimeError(f'Deux Vocals 출력이 하나가 아닙니다: {outputs}')
stem_path = Path(outputs[0])
if not stem_path.is_absolute():
    stem_path = stem_dir / stem_path
source_info, stem_info = sf.info(input_path), sf.info(stem_path)
if abs(source_info.duration-stem_info.duration) > 0.05:
    raise RuntimeError('Deux 전후 음성 길이가 달라 타임스탬프를 그대로 쓸 수 없습니다.')
settings.update(input_duration=source_info.duration, output_duration=stem_info.duration)
temporary = output_path.with_suffix('.partial.wav')
shutil.copyfile(stem_path, temporary)
temporary.replace(output_path)
(config_path.parent / 'separation_info.json').write_text(
    json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
print('Deux BGM 제거 완료:', output_path, flush=True)
'''
DEUX_PATH = RUN / 'separate_deux.py'
DEUX_PATH.write_text(DEUX_SOURCE, encoding='utf-8')
PIPELINE_SOURCE = r'''

import os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
import sys, gc, json, math, subprocess, shutil, csv, zipfile, hashlib
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch, soundfile as sf
from tqdm import tqdm
from qwen_asr import Qwen3ASRModel, Qwen3ForcedAligner
from silero_vad import load_silero_vad, get_speech_timestamps

config_path = Path(sys.argv[1])
cfg = json.loads(config_path.read_text(encoding='utf-8'))
run_dir = config_path.parent
work_dir = run_dir / 'work'
work_dir.mkdir(exist_ok=True)
video_path = Path(cfg['video'])
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
DEVICE, DTYPE = 'cuda:0', torch.float16
ASR_MODEL = 'Qwen/Qwen3-ASR-1.7B'
ALIGN_MODEL = 'Qwen/Qwen3-ForcedAligner-0.6B'
SR, EXPORT_SR = 16000, 48000
MAX_SPEECH_SECONDS, ASR_MERGE_GAP_SECONDS = 25.0, 1.0
ALIGN_RETRY_MAX_SECONDS, ALIGN_RETRY_PAD_SECONDS = 8.0, 0.5
VAD_THRESHOLD = cfg['vad_threshold']
MIN_SPEECH_MS, MIN_SILENCE_MS, SPEECH_PAD_MS = 100, 400, 150
MAX_NEW_TOKENS, ALIGN_TOLERANCE_SECONDS = 768, 0.15
wav_path = work_dir / 'audio_16k.wav'
master_path = work_dir / 'vocals_48k.wav'
source_path = work_dir / 'source_stereo.wav'
vocals_path = work_dir / 'deux_vocals.wav'

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def decode_track(output, sample_rate, input_path=video_path, channels=1, codec='pcm_s16le', source_track=True):
    if output.is_file():
        return
    temporary = output.with_suffix('.partial.wav')
    command = ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-copyts', '-start_at_zero', '-i', str(input_path), '-map',
        f"0:a:{cfg['audio_track']}" if source_track else '0:a:0',
        '-vn', '-af', 'aresample=async=1:first_pts=0', '-ac', str(channels)]
    if sample_rate is not None:
        command += ['-ar', str(sample_rate)]
    command += ['-c:a', codec, str(temporary)]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('원본 트랙 추출 실패: ' + result.stderr)
    temporary.replace(output)

print('1/6 원본 오디오 디코딩', flush=True)
decode_track(source_path, None, channels=2, codec='pcm_f32le')
print('2/6 Deux로 BGM 제거', flush=True)
if not vocals_path.is_file() or not (run_dir / 'separation_info.json').is_file():
    subprocess.run([sys.executable, '-u', str(run_dir / 'separate_deux.py'),
        str(config_path), str(source_path), str(vocals_path)], check=True)
else:
    print('완료된 Deux Vocals를 재사용합니다.', flush=True)
# STT and export are derived from the same separated track.
decode_track(wav_path, SR, input_path=vocals_path, source_track=False)
decode_track(master_path, EXPORT_SR, input_path=vocals_path, source_track=False)
audio, sample_rate = sf.read(wav_path, dtype='float32')
if sample_rate != SR or audio.ndim != 1 or not len(audio) or not np.isfinite(audio).all():
    raise RuntimeError('인식용 오디오 형식이 올바르지 않습니다.')
duration = len(audio)/SR
master_info = sf.info(master_path)
if master_info.samplerate != EXPORT_SR or abs(master_info.duration-duration) > 0.05:
    raise RuntimeError('Deux 추출본과 인식용 오디오의 시간축이 다릅니다.')
print(f'영상 음성 길이: {duration:.2f}초', flush=True)

def merge_speech_segments(segments):
    merged = []
    for segment in segments:
        previous = merged[-1] if merged else None
        if (previous is not None
            and (segment["start"]-previous["end"])/SR <= ASR_MERGE_GAP_SECONDS
            and (segment["end"]-previous["start"])/SR <= MAX_SPEECH_SECONDS):
            previous["end"] = segment["end"]
        else:
            merged.append({"start": int(segment["start"]), "end": int(segment["end"])})
    return merged

segments_path = run_dir / "speech_segments.json"
if segments_path.exists():
    speech_segments = read_json(segments_path)
else:
    vad_model = load_silero_vad()
    raw_segments = get_speech_timestamps(torch.from_numpy(audio), vad_model, sampling_rate=SR,
        threshold=VAD_THRESHOLD, min_speech_duration_ms=MIN_SPEECH_MS,
        max_speech_duration_s=MAX_SPEECH_SECONDS, min_silence_duration_ms=MIN_SILENCE_MS,
        speech_pad_ms=SPEECH_PAD_MS)
    speech_segments = [{"id": i, "start_sample": int(s["start"]), "end_sample": int(s["end"]),
        "start": s["start"]/SR, "end": s["end"]/SR} for i, s in enumerate(merge_speech_segments(raw_segments))]
    del vad_model
    gc.collect()
    save_json(segments_path, speech_segments)
if not speech_segments:
    raise RuntimeError("음성 구간이 없습니다. 선택 오디오와 VAD 설정을 확인하세요.")
print(f"음성 구간 {len(speech_segments)}개 / 합계 {sum(s['end']-s['start'] for s in speech_segments)/60:.2f}분")


print('3/6 일본어 대사 인식', flush=True)

asr_dir = run_dir / "asr_chunks"
asr_dir.mkdir(exist_ok=True)
def chunk_path(directory, segment):
    return directory / f"{segment['id']:05d}.json"
if any(not chunk_path(asr_dir, s).exists() for s in speech_segments):
    asr_model = Qwen3ASRModel.from_pretrained(ASR_MODEL, dtype=DTYPE, device_map=DEVICE,
        attn_implementation="sdpa", max_inference_batch_size=1, max_new_tokens=MAX_NEW_TOKENS)
    print("STT 모델 준비 완료")
else:
    print("저장된 전사 결과가 있어 모델 로드를 생략했습니다.")
print(f"GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
def transcribe_segment(segment):
    path = chunk_path(asr_dir, segment)
    if path.exists():
        return read_json(path)
    print(f"STT 구간 {segment['id']}: {segment['start']:.2f}~{segment['end']:.2f}초", flush=True)
    try:
        result = asr_model.transcribe(audio=(audio[segment["start_sample"]:segment["end_sample"]], SR),
            language="Japanese", return_time_stamps=False)[0]
        record = {**segment, "language": result.language, "text": result.text,
            "empty_transcript": not bool(result.text.strip())}
        save_json(path, record)
        return record
    except Exception as exc:
        save_json(run_dir / "last_error.json", {"stage": "STT", "segment": segment, "error_type": type(exc).__name__})
        raise RuntimeError(f"STT 실패: 구간 {segment['id']}, {segment['start']:.2f}~{segment['end']:.2f}초") from exc
first_transcript = transcribe_segment(speech_segments[0])
print("대사:", first_transcript["text"])
if first_transcript["empty_transcript"]:
    print("확인 필요: 첫 구간 전사가 비었습니다. 음성을 확인하세요.")
transcripts = []
for segment in tqdm(speech_segments, desc="일본어 STT"):
    record = transcribe_segment(segment)
    transcripts.append(record)
    if record["empty_transcript"]:
        print(f"확인 필요: 구간 {record['id']} 전사가 비었습니다.")
save_json(run_dir / "transcripts.json", transcripts)
print("STT 완료:", len(transcripts), "구간 / 빈 전사:", sum(r["empty_transcript"] for r in transcripts))

print('4/6 대사 시간 정렬', flush=True)

if "asr_model" in globals():
    del asr_model
gc.collect()
torch.cuda.empty_cache()
align_dir = run_dir / "aligned_chunks"
align_dir.mkdir(exist_ok=True)
transcripts = read_json(run_dir / "transcripts.json")
if any(r["text"].strip() and not chunk_path(align_dir, r).exists() for r in transcripts):
    align_model = Qwen3ForcedAligner.from_pretrained(ALIGN_MODEL, dtype=DTYPE, device_map=DEVICE,
        attn_implementation="sdpa")
    print("정렬 모델 준비 완료")
else:
    print("저장된 정렬 결과가 있어 모델 로드를 생략했습니다.")
print(f"GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
def align_words(record):
    aligned = align_model.align(audio=(audio[record["start_sample"]:record["end_sample"]], SR),
        text=record["text"], language="Japanese")[0]
    words, previous_start = [], -1.0
    for item in aligned:
        start, end = float(item.start_time), float(item.end_time)
        if (not math.isfinite(start+end) or start < 0 or end < start
            or end > record["end"]-record["start"]+ALIGN_TOLERANCE_SECONDS or start < previous_start):
            raise ValueError(f"비정상 정렬: {item.text!r}, {start}~{end}")
        previous_start = start
        words.append({"segment_id": record["id"], "start": record["start"]+start,
            "end": record["start"]+end, "text": item.text, "display_text": item.text,
            "needs_review": end == start, "alignment_failed": False,
            "timestamp_source": "forced_alignment"})
    if not words or all(w["end"] == w["start"] for w in words):
        return None
    positions, cursor = [], 0
    for word in words:
        position = record["text"].find(word["text"], cursor)
        if position < 0:
            positions = []
            break
        positions.append(position)
        cursor = position+len(word["text"])
    if positions:
        for i, word in enumerate(words):
            left = 0 if i == 0 else positions[i]
            right = positions[i+1] if i+1 < len(words) else len(record["text"])
            word["display_text"] = record["text"][left:right]
    else:
        for word in words:
            word["needs_review"] = True
        print("확인 필요: 문장부호 복원 실패. 원문 전사 JSON을 확인하세요.")
    return words

def make_alignment_fallback(record):
    return {"id": record["id"], "empty_transcript": False, "alignment_failed": True,
        "recovery_complete": False, "words": [{"segment_id": record["id"],
            "start": record["start"], "end": record["end"], "text": record["text"],
            "display_text": record["text"], "needs_review": True,
            "alignment_failed": True, "timestamp_source": "speech_segment"}]}

def align_segment(record):
    path = chunk_path(align_dir, record)
    if path.exists():
        return read_json(path)
    if not record["text"].strip():
        value = {"id": record["id"], "words": [], "empty_transcript": True}
        save_json(path, value)
        return value
    print(f"정렬 구간 {record['id']}: {record['start']:.2f}~{record['end']:.2f}초", flush=True)
    try:
        words = align_words(record)
        if words is None:
            value = make_alignment_fallback(record)
            print(f"확인 필요: 구간 {record['id']} — 전체 정렬 후 해당 구간만 재시도합니다.")
        else:
            value = {"id": record["id"], "words": words, "empty_transcript": False}
            if any(w["needs_review"] for w in words):
                print(f"확인 필요: 구간 {record['id']}의 일부 정렬 결과")
        save_json(path, value)
        return value
    except Exception as exc:
        save_json(run_dir / "last_error.json", {"stage": "alignment", "segment_id": record["id"],
            "error_type": type(exc).__name__})
        raise RuntimeError(f"시간 정렬 실패: 구간 {record['id']}. 전사와 음성을 확인하세요.") from exc

def release_alignment_model():
    global align_model
    if "align_model" in globals():
        del align_model
    gc.collect()
    torch.cuda.empty_cache()

def split_original_transcript(text, retry_texts):
    # Punctuation may differ; every spoken character must match the saved original.
    positions = [i for i, char in enumerate(text) if char.isalnum()]
    original = "".join(text[i] for i in positions)
    normalized = ["".join(c for c in value if c.isalnum()) for value in retry_texts]
    if not original or any(not value for value in normalized) or "".join(normalized) != original:
        return None
    parts, left, cursor = [], 0, 0
    for i, value in enumerate(normalized):
        cursor += len(value)
        right = positions[cursor] if i+1 < len(normalized) else len(text)
        parts.append(text[left:right])
        left = right
    return parts

for record in tqdm(transcripts, desc="대사 시간 정렬"):
    align_segment(record)
release_alignment_model()
failed = [r for r in transcripts if read_json(chunk_path(align_dir, r)).get("alignment_failed")
    and not read_json(chunk_path(align_dir, r)).get("recovery_complete")]
retry_dir = run_dir / "alignment_retries"
retry_dir.mkdir(exist_ok=True)
retry_candidates = []
if failed:
    print(f"시간 정렬 실패 {len(failed)}개 구간만 나누어 다시 확인합니다.")
    vad_model = load_silero_vad()
    for record in failed:
        retry_path = chunk_path(retry_dir, record)
        if retry_path.exists():
            retry = read_json(retry_path)
        else:
            pieces = get_speech_timestamps(
                torch.from_numpy(audio[record["start_sample"]:record["end_sample"]]),
                vad_model, sampling_rate=SR, threshold=VAD_THRESHOLD,
                min_speech_duration_ms=MIN_SPEECH_MS,
                max_speech_duration_s=ALIGN_RETRY_MAX_SECONDS,
                min_silence_duration_ms=MIN_SILENCE_MS, speech_pad_ms=SPEECH_PAD_MS)
            retry = {"id": record["id"], "parts": [{"id": record["id"],
                "start_sample": record["start_sample"]+int(p["start"]),
                "end_sample": record["start_sample"]+int(p["end"]),
                "start": (record["start_sample"]+int(p["start"]))/SR,
                "end": (record["start_sample"]+int(p["end"]))/SR} for p in pieces]}
            save_json(retry_path, retry)
        retry_candidates.append((record, retry))
    del vad_model
    gc.collect()
    if any(len(retry["parts"]) > 1 and any("retry_text" not in p for p in retry["parts"])
        for _, retry in retry_candidates):
        asr_model = Qwen3ASRModel.from_pretrained(ASR_MODEL, dtype=DTYPE, device_map=DEVICE,
            attn_implementation="sdpa", max_inference_batch_size=1, max_new_tokens=MAX_NEW_TOKENS)
        try:
            for record, retry in tqdm(retry_candidates, desc="실패 구간의 대사 경계 확인"):
                if len(retry["parts"]) <= 1:
                    continue
                for part in retry["parts"]:
                    if "retry_text" not in part:
                        result = asr_model.transcribe(
                            audio=(audio[part["start_sample"]:part["end_sample"]], SR),
                            language="Japanese", return_time_stamps=False)[0]
                        part["retry_text"] = result.text
                        save_json(chunk_path(retry_dir, record), retry)
        finally:
            del asr_model
            gc.collect()
            torch.cuda.empty_cache()
    align_model = Qwen3ForcedAligner.from_pretrained(ALIGN_MODEL, dtype=DTYPE, device_map=DEVICE,
        attn_implementation="sdpa")
    try:
        for record, retry in tqdm(retry_candidates, desc="실패 구간 시간 재정렬"):
            # A small context margin can recover short utterances without guessing text boundaries.
            padded = {**record,
                "start_sample": max(0, record["start_sample"]-round(ALIGN_RETRY_PAD_SECONDS*SR)),
                "end_sample": min(len(audio), record["end_sample"]+round(ALIGN_RETRY_PAD_SECONDS*SR))}
            padded.update(start=padded["start_sample"]/SR, end=padded["end_sample"]/SR)
            recovered = align_words(padded)
            if recovered is not None and any(w["start"] < record["start"] or w["end"] > record["end"]
                for w in recovered):
                recovered = None
            if recovered is None and len(retry["parts"]) > 1:
                texts = split_original_transcript(record["text"], [p["retry_text"] for p in retry["parts"]])
                if texts is not None:
                    collected = []
                    for part, text in zip(retry["parts"], texts):
                        part_words = align_words({**part, "text": text})
                        if part_words is None:
                            break
                        collected.extend(part_words)
                    else:
                        recovered = collected
            value = make_alignment_fallback(record)
            value["recovery_complete"] = True
            if recovered is not None:
                for word in recovered:
                    word["timestamp_source"] = "retry_alignment"
                value.update(words=recovered, alignment_failed=False)
                print(f"구간 {record['id']}: 시간 정렬 복구 완료")
            else:
                print(f"확인 필요: 구간 {record['id']} — 원문 보존, 화자 미배정")
            save_json(chunk_path(align_dir, record), value)
    finally:
        release_alignment_model()
words = [word for record in transcripts for word in read_json(chunk_path(align_dir, record))["words"]]
save_json(run_dir / "aligned_words.json", words)
print(f"정렬 완료: {len(words)}개 / 확인 필요: {sum(w['needs_review'] for w in words)}개")
print(f"해제 후 GPU 사용 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")


print('5/6 BGM 제거 음성 구간 전체·검수 목록 생성', flush=True)

clips = []
for record in transcripts:
    aligned = read_json(chunk_path(align_dir, record))
    reasons = []
    if not record['text'].strip():
        reasons.append('empty_transcript')
    elif aligned.get('alignment_failed') or any(w['needs_review'] for w in aligned.get('words', [])):
        reasons.append('alignment_uncertain')
    # Keep the complete audio given to STT, together with its complete transcript.
    # Word alignment remains metadata; its final word no longer defines the WAV end.
    clips.append(dict(start=record['start'], end=record['end'], text=record['text'],
        segment_id=record['id'], timestamp_source='speech_segment',
        alignment_failed=bool(aligned.get('alignment_failed')), review_reasons=reasons))

clips.sort(key=lambda row: (row['start'], row['end']))
dataset = run_dir / cfg['dataset_name']
wave_dir = dataset / 'wav'
metadata_dir = dataset / 'metadata'
wave_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(exist_ok=True)
rows = []
with sf.SoundFile(master_path) as master:
    for index, clip in enumerate(tqdm(clips, desc='Deux WAV 음성 구간 전체 추출')):
        left = max(0.0, clip['start'])
        right = min(master.frames/EXPORT_SR, clip['end'])
        first = max(0, min(master.frames, math.floor(left*EXPORT_SR)))
        last = max(0, min(master.frames, math.ceil(right*EXPORT_SR)))
        if last <= first:
            raise RuntimeError(f'추출 길이가 0입니다: {clip}')
        master.seek(first)
        waveform = master.read(last-first, dtype='int16', always_2d=False)
        name = f"{cfg['dataset_name']}_{index+1:05d}.wav"
        output = wave_dir / name
        temporary = output.with_suffix('.partial.wav')
        sf.write(temporary, waveform, EXPORT_SR, subtype='PCM_16')
        temporary.replace(output)
        seconds = len(waveform)/EXPORT_SR
        reasons = list(clip['review_reasons'])
        if seconds < 0.5:
            reasons.append('shorter_than_training_minimum')
        if seconds > 20:
            reasons.append('longer_than_training_maximum')
        rows.append(dict(audio_file=name, text=clip['text'],
            start_seconds=round(clip['start'], 6), end_seconds=round(clip['end'], 6),
            clip_start_seconds=first/EXPORT_SR, clip_end_seconds=last/EXPORT_SR,
            duration_seconds=seconds, source_video=video_path.name, audio_track=cfg['audio_track'],
            segment_id=clip['segment_id'], timestamp_source=clip['timestamp_source'],
            alignment_failed=clip['alignment_failed'], needs_review=True,
            review_reasons=';'.join(reasons), speaker='', keep='', corrected_text=''))

def write_csv(path, entries, fields):
    with path.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(entries)

write_csv(dataset/'captions.csv', rows, ['audio_file', 'text'])
review_fields = ['audio_file', 'text', 'start_seconds', 'end_seconds', 'clip_start_seconds',
    'clip_end_seconds', 'duration_seconds', 'source_video', 'audio_track', 'segment_id',
    'timestamp_source', 'alignment_failed', 'needs_review', 'review_reasons',
    'speaker', 'keep', 'corrected_text']
write_csv(dataset/'review.csv', rows, review_fields)
save_json(metadata_dir/'clips.json', rows)
for filename in ['transcripts.json', 'aligned_words.json', 'speech_segments.json', 'separation_info.json']:
    shutil.copyfile(run_dir/filename, metadata_dir/filename)
save_json(metadata_dir/'source_info.json', dict(config=cfg, timeline='video start at zero',
    export_sample_rate=EXPORT_SR, channels=1, subtype='PCM_16',
    audio_source='Deux Vocals', clip_boundary=cfg['clip_boundary'],
    media=json.loads(subprocess.check_output(['ffprobe', '-v', 'error', '-show_streams',
        '-show_format', '-of', 'json', str(video_path)], text=True)),
    packages={name: version(name) for name in ['qwen-asr', 'silero-vad', 'torch', 'transformers', 'audio-separator']}))

def srt_time(seconds):
    milliseconds = round(seconds*1000)
    hours, milliseconds = divmod(milliseconds, 3600000)
    minutes, milliseconds = divmod(milliseconds, 60000)
    seconds, milliseconds = divmod(milliseconds, 1000)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d},{milliseconds:03d}'

srt_rows = []
for row in rows:
    if row['end_seconds'] > row['start_seconds'] and row['text'].strip():
        srt_rows.append(f"{len(srt_rows)+1}\n{srt_time(row['start_seconds'])} --> "
            f"{srt_time(row['end_seconds'])}\n{row['text']}\n")
(dataset/'dialogues.srt').write_text('\n'.join(srt_rows), encoding='utf-8')
(dataset/'검수안내.txt').write_text(
    '로컬 검수 순서\n'
    '1. review.csv에서 audio_file에 해당하는 wav 파일을 듣고 화자와 대사를 확인합니다.\n'
    '2. speaker, keep, corrected_text는 검수 기록용 빈 칸입니다. 자동 적용되지 않습니다.\n'
    '3. start_seconds/end_seconds는 STT에 입력한 음성 구간 전체의 경계입니다. clip_start_seconds/clip_end_seconds는 이를 샘플 단위로 맞춘 실제 추출 경계입니다. 모두 원본 영상 시작 기준 초입니다.\n'
    '4. 모든 WAV는 timestamp_source=speech_segment입니다. 말끝을 보존하도록 단어 정렬 끝에서 자르지 않으며, aligned_words.json에는 단어 정렬 시각을 따로 보존합니다.\n'
    '5. 모든 행은 검수 대상입니다. review_reasons가 비어 있어도 오인식, 다른 화자, BGM을 확인하세요.\n'
    '6. 채택한 WAV만 남기고 captions.csv의 audio_file,text를 해당 음성과 교정된 일본어 대사에 맞춥니다.\n'
    '7. 현재 Irodori 노트북 기본값은 0.5~20초이며 최소 20행을 요구합니다. 긴 후보는 음성과 대사를 함께 나누세요.\n'
    '8. 최종 ZIP에는 captions.csv를 하나만 넣고 같은 폴더 아래 wav/를 유지합니다.\n'
    '9. Deux Vocals에서 추출했습니다. 잔여 BGM·효과음·음성 손상·다른 화자를 확인하세요. 에키드나 자동 식별은 하지 않았습니다.\n', encoding='utf-8')
report = dict(clips=len(rows), total_seconds=sum(row['duration_seconds'] for row in rows),
    flagged_clips=sum(bool(row['review_reasons']) for row in rows),
    empty_transcripts=sum(not row['text'].strip() for row in rows),
    unconfirmed_speaker_clips=len(rows), source_video_sha256=cfg['video_sha256'])
save_json(metadata_dir/'dataset_report.json', report)
print('6/6 ZIP 생성', flush=True)
zip_path = run_dir / f"{cfg['dataset_name']}_first_pass.zip"
temporary_zip = zip_path.with_suffix('.partial.zip')
with zipfile.ZipFile(temporary_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(dataset.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(run_dir).as_posix())
temporary_zip.replace(zip_path)
save_json(run_dir/'result.json', dict(zip=str(zip_path), dataset=str(dataset), report=report))
print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
print('완료:', zip_path, flush=True)

'''
PIPELINE_PATH = RUN / 'build_dataset.py'
PIPELINE_PATH.write_text(PIPELINE_SOURCE, encoding='utf-8')
run_process([PY, '-u', PIPELINE_PATH, CONFIG_PATH])
RESULT = json.loads((RUN / 'result.json').read_text(encoding='utf-8'))
print('다운로드 준비:', RESULT['zip'])

In [ ]:
#@title 4. 결과 확인 · ZIP 다운로드
import csv
from IPython.display import display, Audio

RESULT = json.loads((RUN / 'result.json').read_text(encoding='utf-8'))
DATASET = Path(RESULT['dataset'])
ZIP_PATH = Path(RESULT['zip'])
print(json.dumps(RESULT['report'], ensure_ascii=False, indent=2))
with (DATASET / 'review.csv').open(encoding='utf-8-sig', newline='') as stream:
    preview_rows = list(csv.DictReader(stream))
for row in preview_rows[:3]:
    print(f"{row['audio_file']} | {float(row['start_seconds']):.2f}~"
          f"{float(row['end_seconds']):.2f}초 | {row['text']} | {row['review_reasons']}")
    display(Audio(filename=str(DATASET / 'wav' / row['audio_file'])))
print(f'ZIP: {ZIP_PATH.stat().st_size/1024**2:.1f} MiB')
print('자동 다운로드가 막히면 코랩 왼쪽 파일 메뉴에서 위 ZIP을 내려받으세요.')
files.download(str(ZIP_PATH))

## 로컬 검수 후 학습 연결

ZIP을 풀고 `review.csv`와 WAV를 비교하여 에키드나의 발화를 고릅니다. 최종 `captions.csv`에는 채택한 WAV 파일명과 교정한 일본어 대사만 남깁니다. `review.csv`의 `keep`이나 `corrected_text`를 바꿔도 `captions.csv`에는 자동 반영되지 않습니다.

완성된 `echidna/captions.csv`와 `echidna/wav/`를 ZIP으로 묶어 Irodori 노트북의 `dataset_zip_path`에 지정하세요. 현재 학습 노트북은 폴더 이름을 고정하지 않고 `captions.csv`를 검색합니다. 이 1차 ZIP은 검수 후보이며, 학습 품질은 직접 확인한 최종 음성과 대사에 따라 달라집니다.

문제 확인: 설치 실패는 셀 1 로그, 잘못된 언어는 셀 2의 트랙 선택, 누락은 VAD·전사 결과, 잘못된 시간은 `metadata/speech_segments.json`, `metadata/aligned_words.json`과 `review.csv`를 확인합니다. BGM 제거 설정은 `metadata/separation_info.json`에 기록됩니다. 작은 목소리가 누락되면 `vad_threshold`를 낮춰 새 설정으로 실행할 수 있습니다. 파일 업로드·전사·ZIP 다운로드까지의 실제 코랩 GPU 실행은 제작 환경에서 확인하지 못했습니다.